## Load libraries

In [50]:
import torch
import timm
import os
import sys
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
import json
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

# Device configuration - uses GPU if available, falls back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


## Load model as a TorchScript file (`.pt`)

In [51]:
# Model path
data_root = "C:/Users/pdeschepper/OneDrive - Institute of Natural Sciences/Desktop"
model_filename = "best.pt"  
model_path = os.path.join(data_root, model_filename)

# We know the model is of type : dict, explore structure
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
print("arch:", checkpoint['arch'])
print("encoder:", checkpoint['encoder'])
print("weights:", checkpoint['weights'])
print("size:", checkpoint['size'])
print("best:", checkpoint['best'])
print("seed:", checkpoint['seed'])
print("run_id:", checkpoint['run_id'])

state_dict = checkpoint['model']
print("\nnum params in state_dict:", len(state_dict))
print("first 10 keys:", list(state_dict.keys())[:10])

# Find the segmentation head's final conv layer to read off output channels -> should be ssingle-channel sigmoid output
for k, v in state_dict.items():
    if "segmentation_head" in k or ("head" in k and "weight" in k):
        print(k, v.shape)


model = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b0",
    encoder_weights=None,   # loading your trained weights below, not fresh pretrained ones
    in_channels=3,
    classes=1,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval()
model.to(device)
print("✅ Loaded")

# Get the correct preprocessing config for this specific encoder/weights combo
model_cfg = timm.create_model("tf_efficientnet_b0.ns_jft_in1k", pretrained=False).default_cfg
print("mean:", model_cfg['mean'])
print("std:", model_cfg['std'])

arch: unetpp
encoder: timm-efficientnet-b0
weights: noisy-student
size: 512
best: {'iou_fg': 0.8783774719103743, 'leakage': 0.05858514637828649, 'boundary_iou': 0.7075640172092581, 'iou_mean': 0.9337842230109024, 'stage': 'stage1', 'epoch': 40}
seed: 42
run_id: 20260606-152431-seed42-a45b65a

num params in state_dict: 492
first 10 keys: ['encoder.conv_stem.weight', 'encoder.bn1.weight', 'encoder.bn1.bias', 'encoder.bn1.running_mean', 'encoder.bn1.running_var', 'encoder.bn1.num_batches_tracked', 'encoder.blocks.0.0.conv_dw.weight', 'encoder.blocks.0.0.bn1.weight', 'encoder.blocks.0.0.bn1.bias', 'encoder.blocks.0.0.bn1.running_mean']
encoder.conv_head.weight torch.Size([1280, 320, 1, 1])
segmentation_head.0.weight torch.Size([1, 16, 3, 3])
segmentation_head.0.bias torch.Size([1])
Missing keys: []
Unexpected keys: []
✅ Loaded
mean: (0.485, 0.456, 0.406)
std: (0.229, 0.224, 0.225)


## Preprocessing + inference loop, updated for this model (classes=1 → sigmoid, size=512 confirmed, no argmax needed):

In [52]:
# --- Preprocessing configuration ---
INPUT_SIZE = (checkpoint['size'], checkpoint['size'])  # 512x512, straight from the checkpoint
MEAN = np.array(model_cfg['mean'], dtype=np.float32)   # from the timm config above
STD = np.array(model_cfg['std'], dtype=np.float32)


def preprocess_image(img_original, size=INPUT_SIZE):
    """Resize + normalize a PIL RGB image to a CHW tensor for model input."""
    img_resized = img_original.resize(size)
    img_arr = np.array(img_resized, dtype=np.float32) / 255.0  # HWC, [0, 1]
    img_arr = (img_arr - MEAN) / STD
    img_tensor = torch.from_numpy(img_arr.transpose(2, 0, 1)).unsqueeze(0)  # 1, C, H, W
    return img_tensor.float()


# Configuration
image_folder = "C:/Users/pdeschepper/OneDrive - Institute of Natural Sciences/Desktop/PERSONAL/DeepLearning/ImageSegmentation/Snakes_ImageSegmentation_keras/Vipera_segmentation_test_dataset/" 
output_folder = os.path.join(image_folder, "Extracted_snakes_pytorch")

os.makedirs(output_folder, exist_ok=True)
print(f"Output folder created/verified: {output_folder}\n")

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
image_files = [f for f in os.listdir(image_folder)
               if os.path.splitext(f)[1].lower() in image_extensions
               and not os.path.isdir(os.path.join(image_folder, f))]

print(f"Found {len(image_files)} images to process\n")

output_csv_path = os.path.join(output_folder, "class_coverage.csv")
with open(output_csv_path, "w") as coverage:
    coverage.write("photo_ID,class_coverage\n")
    for idx, img_name in enumerate(image_files, 1):
        try:
            img_path = os.path.join(image_folder, img_name)

            img_original = Image.open(img_path).convert('RGB')
            original_size = (img_original.width, img_original.height)
            img_array = np.array(img_original)

            img_input = preprocess_image(img_original).to(device)

            print(f"[{idx}] Processing {img_name}...", end=" ")

            with torch.no_grad():
                mask_output = model(img_input)  # shape: [1, 1, H, W]

            mask_prob = torch.sigmoid(mask_output)[0, 0].cpu().numpy()
            mask_pred = (mask_prob > 0.5).astype(np.uint8)

            mask_percentage = (np.sum(mask_pred == 1) / mask_pred.size) * 100
            print(f"(class 1 coverage: {mask_percentage:.1f}%)", end=" ")

            coverage.write(f"{img_name},{mask_percentage:.2f}\n")

            mask_pil = Image.fromarray((mask_pred * 255).astype(np.uint8))
            mask_pil = mask_pil.resize(original_size, Image.NEAREST)
            mask_array = np.array(mask_pil).astype(np.float32) / 255.0

            mask_expanded = np.stack([mask_array] * 3, axis=2)
            extracted_rgb = (img_array * mask_expanded).astype(np.uint8)

            extracted = np.zeros((original_size[1], original_size[0], 4), dtype=np.uint8)
            extracted[:, :, :3] = extracted_rgb
            extracted[:, :, 3] = (mask_array * 255).astype(np.uint8)

            output_name = os.path.splitext(img_name)[0] + '.png'
            output_path = os.path.join(output_folder, output_name)
            result_img = Image.fromarray(extracted, 'RGBA')
            result_img.save(output_path)

            print("✓ Saved")

        except Exception as e:
            print(f"ERROR processing {img_name}: {e}")
            continue

print(f"\nProcessing complete! Results saved to: {output_folder}")

Output folder created/verified: C:/Users/pdeschepper/OneDrive - Institute of Natural Sciences/Desktop/PERSONAL/DeepLearning/ImageSegmentation/Snakes_ImageSegmentation_keras/Vipera_segmentation_test_dataset/Extracted_snakes_pytorch

Found 10 images to process

[1] Processing 541382491.jpg... (class 1 coverage: 33.1%) 

<positron-console-cell-52>:69: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


✓ Saved
[2] Processing 541698315.jpg... (class 1 coverage: 27.3%) ✓ Saved
[3] Processing 541698521.jpg... (class 1 coverage: 25.5%) ✓ Saved
[4] Processing 544934383.jpg... (class 1 coverage: 19.9%) ✓ Saved
[5] Processing 544934561.jpg... (class 1 coverage: 16.8%) ✓ Saved
[6] Processing 559768117.jpg... (class 1 coverage: 7.2%) ✓ Saved
[7] Processing 561468843.jpg... (class 1 coverage: 30.4%) ✓ Saved
[8] Processing 561842903.jpg... (class 1 coverage: 39.8%) ✓ Saved
[9] Processing 568664402.jpg... (class 1 coverage: 39.2%) ✓ Saved
[10] Processing 576802297.jpg... (class 1 coverage: 58.3%) ✓ Saved

Processing complete! Results saved to: C:/Users/pdeschepper/OneDrive - Institute of Natural Sciences/Desktop/PERSONAL/DeepLearning/ImageSegmentation/Snakes_ImageSegmentation_keras/Vipera_segmentation_test_dataset/Extracted_snakes_pytorch
